<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>Semantic Networks</font><br>
<font size=20 color=#00c382>Medieval Weapons</font><br>

|**Assessment Code:** |DAT-6002|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S1_26 *CHECK*|
|**Course Tutor:**|Imtiaz ****|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**


---

# Initialise Notebook

In [10]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [11]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime
import re

# Requests
import requests

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
%matplotlib inline

# Colorama
from colorama import Fore, Back, Style

# NetworkX
import networkx as nx

# PyVis
from pyvis.network import Network

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [12]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [13]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [14]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [15]:
internet_required = False

In [16]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Define Semantic Network Class

This class defines a semantic network for any data. The primary purpose is to allow simple gathering of data and to then visualise the network.

In [ ]:
# Define Class SemanticNetwork
class SemanticNetwork:

    # Constructor
    def __init__(self, root_node="RootNode"):

        # Create graph object
        self.G = nx.DiGraph()
        self.root = root_node
        
        # Define palette (from https://coolors.co, using shades option)
        self.palette = {0:"#C3ADFF", 1:"#9670ff", 2:"#7847ff", 3:"#5A1FFF", 4:"#4100F5", 5:"#3100B8", 6:"#22007C"}
        self.bgcolour = "black"
        self.node_text_colour = "white"
        self.edge_label_colour = "#F0D90A"
        self.edge_colour_structural = "#c4830b81"
        self.edge_colour_property = "#7847ff81"

    # Add structural relationships to the network (IS-A type)
    def add_relation(self, node, parent):
        self.G.add_edge(node, parent, label="is_a")

    # Add property relationships to the network (HAS-PART type)
    def add_property(self, node, label, value):
        self.G.add_edge(node, value, label=label)
    
    # Calculates distance from the root node
    # Used for dynamic scaling and chosing colours from palette
    def get_node_level(self, node):
        try:
            return nx.shortest_path_length(self.G.to_undirected(), source=node, target=self.root)
        except:
            return None

    # Export Network to json file 
    def export_to_json(self, filename="network.json"):
        #data = nx.edges(self.G)
        data = nx.node_link_data(self.G)    # deprecated...

        # *** Note: WRAP WITH A TRY 
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)

        print_output(f"Successfully saved", f"{len(self.G.nodes)} nodes to {filename}")

    # Import Network from json file 
    def import_from_json(self, filename="network.json"):
        
        # *** Note: WRAP WITH A TRY 
        with open(filename, 'r') as f:
            data = json.load(f)

        # Destructive
        self.G = nx.node_link_graph(data)
        print_output(f"Successfully loaded", f"{len(self.G.nodes)} nodes from {filename}")

    def print_levels (self):

        for s in self.G.nodes:
            print (s)
            spacer = {s: 0}
            for prereq, target in nx.dfs_edges(self.G, s):
                spacer[target] = spacer[prereq] + 2
                print (f'{spacer}+-{target}')
            print ('')


    # Create a dynamic visualisation of the network using PyVis, saved as HTML file
    def draw(self, filename="dynamic_network_diagram.html"):

        # Initiate the PyVis Network
        net = Network(height="1000", width="100%", bgcolor=self.bgcolour, font_color=self.node_text_colour, directed=True)
        
        #dist = {n: self.get_node_level(n) for n in self.G.nodes() if self.get_node_level(n) is not None}

        # Add Nodes 
        for node in self.G.nodes():
            #level = dist.get(node, 99)
            level = self.get_node_level(node) or 99
            if node == self.root:
                level = 0
            
            color = self.palette.get(level, "#D3D3D3")


            
            # Dynamic Scaling of nodes and fonts
            if level==0:
                dynamic_node_size = 120
                dynamic_font_size = 120
            else:
                dynamic_font_size = max(80 - (level * 8), 25) if level != 99 else 20
                dynamic_node_size = max(70 - (level * 7), 15) if level != 99 else 10

            
            net.add_node(
                node, 
                label=node, 
                color=color, 
                size = dynamic_node_size,
                font={'size': dynamic_font_size, 'face': 'arial', 'color': self.node_text_colour}
            )

        # Add Edges
        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            # Check if it is a structural label, as in an 'is_a' type relationship
            is_structural = label in 'is_a'
            net.add_edge(
                source, 
                target, 
                label=label,
                width=5 if is_structural else 2,
                color= self.edge_colour_structural if is_structural else self.edge_colour_property,
                font={'color': self.edge_label_colour, 'size': 20, 'strokeWidth': 0}
            )

        # Define options for network
        options = {
            "nodes": {
                "font": {
                    "size": 30 
                }
            },
            "edges": {
                "font": {
                    "size": 18,
                    "strokeWidth": 0
                },
                "color": {"inherit": False}
            },
            "physics": {
                "enabled": True,
                "barnesHut": {
                    "gravitationalConstant": -13000, # Push nodes apart
                    "springLength": 300 # Lengthen the edges
                },
                "solver": "barnesHut"
            },
            "interaction": {
                "hideEdgesOnDrag": False,
                "hideEdgesOnZoom": False,
                "zoomView": True
            }
        }
        
        # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)


    # The network can export itself to Prolog
    # Requires a rules file to be appended to the output
    def export_to_prolog (self, filename="prolog_file.pl",title = "Semantic Network",rules_filename='rules.pl'):

        # Separate storage for different types of facts
        is_a_facts = []
        has_a_facts = []
        # Empty File
        prolog_file = []

        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            src = f"{source.lower().replace(' ','_')}"
            tgt = f"{target.lower().replace(' ','_')}"
            label = label.lower().replace('-','_')
                
            # Grouping types together
            if label in ["is_a", "instance_of", "type_of"]:
                is_a_facts.append(f"is_a( {src}, {tgt} ).")
            else:
                # Special case when there is a 'from x to y' value
                if '-' in tgt:
                    tgt = f"{tgt.replace('-','_').replace('_',',')}"

                has_a_facts.append(f"has_a( {src}, {label}({tgt}) ).")

        # Sort the facts
        is_a_facts.sort()
        has_a_facts.sort()

        # Construct the file content in blocks
        header_block = [
            "%\n",
            f"% {title} Represented in Prolog\n",
            "%\n"
            f"% Auto generated from SemanticNetwork Class, by Tom Rowan",
            "%\n"
        ]
        
        prolog_file.append("\n% is_a facts\n")
        prolog_file.extend(is_a_facts)
        
        prolog_file.append("\n% has_a facts\n")
        prolog_file.extend(has_a_facts)

        # Append the rules
        with open(rules_filename, 'r') as f:
            rules_data = f.read()

        prolog_file.append(rules_data)

        # Save to file
        with open(filename, 'w') as f:
            for line in prolog_file:
                f.write(str(line)+'\n')
        
        print_output("Wrote File",f"Prolog written to {filename}.")


    # The network can be imported from a prolog file provided it follows the right structure
    #
    #   File must be formatted as follows:
    #       is_a(mammal, animal)
    #       is_a(canine, mammal)
    #       ..
    #       has_a(canine, makes_noise(bark))
    #       has_a(canine, makes_noise(growl))
    #       has_a(canine, legs(4))
    #
    def import_from_prolog (self, filename='prolog_file.pl'):

        # Empty lists
        is_a_list = []
        has_a_list = []

        # Regex patterns
        is_a_pattern = r"^is_a\(\s*(\w+),\s*(\w+)\s*\)\."
        has_a_pattern = r"^has_a\(\s*(\w+),\s*(\w+)\((.*?)\)\s*\)\."

        with open(filename, 'r') as file:
            data = file.read()

            # Extracting is_a facts
            is_a_list = re.findall(is_a_pattern, data, re.MULTILINE)
            
            # Extracting has_a facts
            has_a_list = re.findall(has_a_pattern, data, re.MULTILINE)

        # Add elements to the network
        for fact in is_a_list:
            entity = fact[0]
            parent = fact[1]
            self.add_relation(entity, parent)
    
        for fact in has_a_list:
            entity = fact[0]
            property = fact[1]
            value = fact[2]
            self.add_property(entity, property, value)

        print_output ("Imported network from prolog file", filename)
        print(f" - Added {str(len(is_a_list))} IS_A facts")
        print(f" - Added {str(len(has_a_list))} HAS_A facts")


    def get_inherited_attribute(self, node, attribute_label):
        current = node
        while current:
            # Check current node for the specific property
            for _, neighbor, data in self.G.out_edges(current, data=True):
                if data.get('label') == attribute_label:
                    return neighbor
            # Climb up the hierarchy
            parents = [target for _, target, data in self.G.out_edges(current, data=True) 
                       if data.get('label') in self.structural_labels]
            current = parents[0] if parents else None
        return "N/A"


    def generate_report(self):
        report_data = []
        # Filter for 'Concept' nodes (those with a hierarchical level)
        concept_nodes = [n for n in self.G.nodes() if self.get_node_level(n) is not None]
        
        for node in concept_nodes:
            report_data.append({
                "Entity": node,
                "Level": self.get_node_level(node),
                "Damage": self.get_inherited_attribute(node, "causesDamage"),
                "Material": self.get_inherited_attribute(node, "primaryMaterial"),
                "Origin": self.get_inherited_attribute(node, "historicalOrigin"),
                "Era": self.get_inherited_attribute(node, "eraUsed")
            })
        return pd.DataFrame(report_data).sort_values("Level")


# Populate Network

The network can be populated manually, for example:

```
# Create new network object (Level 0)
weapons_net = SemanticNetwork(root_node="weapon")

# Add weapons to the hierarchy

### Level 1 - Categories
# Read as follows: a melee_weapon is_a weapon
weapons_net.add_relation("melee_weapon", "weapon")

### Level 2 - Categories
# Melee Weapon Categories
weapons_net.add_relation("bladed_hand_weapon", "melee_weapon")

### Level 3 - Categories
# Bladed Hand Weapon Categories
weapons_net.add_relation("sword", "bladed_hand_weapon")

# Properties of a Generic Sword
# Read as follows: a sword has_a (has_part) pommel
weapons_net.add_property("sword", "has_part", "pommel")
weapons_net.add_property("sword", "has_part", "hilt")
weapons_net.add_property("sword", "has_part", "blade")
weapons_net.add_property("sword", "causes_damage", "cut")
weapons_net.add_property("sword", "primary_material", "steel" )


### Level 4 - Weapons and properties
# Swords
weapons_net.add_relation("long_sword", "sword")

### Level 5 (Specific Instances)
weapons_net.add_relation("the_wallace_sword", "long_sword")

weapons_net.add_property("the_wallace_sword", "weight_kg", "2.7") 
weapons_net.add_property("the_wallace_sword", "historical_origin", "scotland")
weapons_net.add_property("the_wallace_sword", "era_used", "late_13th_century")
```

It can also be populated from a prolog file:

In [18]:
# Create network
weapons_net = SemanticNetwork (root_node='weapon')

# Import Prolog File
weapons_net.import_from_prolog(filename='2-Semantic-Network-Weapons-backup.pl')

# Export to JSON to share with others :-)
weapons_net.export_to_json(filename='3-Mediaval_Weapons_Network.json')

# Draw a dynamic PyVis diagram
weapons_net.draw(filename='4-Semantic-Network-Diagram.html')

Imported network from prolog file: 2-Semantic-Network-Weapons-backup.pl
 - Added 72 IS_A facts
 - Added 58 HAS_A facts
Successfully saved: 119 nodes to 3-Mediaval_Weapons_Network.json


# Generate Visualisations

In [19]:



# Create the interactive file
filename = "medieval_weapons_semantic_network_" + now_datetime_timestamp + ".html"
weapons_net.draw(filename)

In [20]:
# Save the network out
filename = "medieval_weapons_semantic_network_" + now_datetime_timestamp + ".json"
weapons_net.export_to_json(filename)

# Import from json
# weapons_net.import_from_json(filename)

Successfully saved: 119 nodes to medieval_weapons_semantic_network_20260223-1538.json


## Prolog to SN